# DocuSense: retrieval experiment analysis
Run `python scripts/benchmark.py --semantic` first. Select the project's `.venv` kernel. These are development examples, not a held-out test set. No model training or external API is involved in this notebook.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
ROOT = Path.cwd() if (Path.cwd() / 'reports').exists() else Path.cwd().parent
results = pd.read_csv(ROOT / 'reports' / 'ablation.csv')
results

,engine,chunk_words,overlap,chunks,index_ms,questions,recall@1,recall@3,recall@5,mrr@5,retrieval_ms
0,tfidf,80,15,26,41.5385,18,0.888889,0.944444,0.944444,0.907407,1.589650
1,semantic,80,15,26,1145.8326,18,1.000000,1.000000,1.000000,1.000000,67.695539
2,tfidf,160,30,13,9.2966,18,0.888889,0.944444,0.944444,0.907407,1.664667
3,semantic,160,30,13,766.8642,18,1.000000,1.000000,1.000000,1.000000,49.510900
4,tfidf,220,40,9,7.4994,18,0.888889,0.944444,0.944444,0.907407,3.490367
5,semantic,220,40,9,373.5140,18,1.000000,1.000000,1.000000,1.000000,37.870544


## Compare a lexical baseline with pretrained semantic retrieval
Keep page labels fixed while varying windows. Recall uses distinct pages in the numerator but actual chunk ranks. MRR is truncated at five. Do not compare cosine scores as calibrated probabilities.

In [2]:
long = results.melt(id_vars=['engine', 'chunk_words'], value_vars=['recall@1', 'recall@3', 'recall@5', 'mrr@5'], var_name='metric', value_name='score')
px.bar(long, x='chunk_words', y='score', color='engine', facet_col='metric', barmode='group', range_y=[0, 1.05])

## Inspect failures instead of hiding them in averages
For each miss, distinguish retrieval errors from incomplete labels. Inspect the relevant source page and compare the returned chunk IDs. Larger chunks and repeated overlapping passages can change ranks.

In [3]:
queries = pd.read_csv(ROOT / 'reports' / 'queries_tfidf_160.csv')
queries.loc[queries['recall@1'] < 1, ['question', 'recall@1', 'recall@5', 'mrr@5', 'retrieved_ids', 'relevant_ids']]

,question,recall@1,recall@5,mrr@5,retrieved_ids,relevant_ids
9,Which metadata makes a passage traceable to it...,0.0,1.0,0.333333,44cbadf1bcf59b80c037246dcd3f8d5eab3bf95c0c83b7...,44cbadf1bcf59b80c037246dcd3f8d5eab3bf95c0c83b7...
14,What information is needed to reproduce a retr...,0.0,0.0,0.000000,7b5a158b064d94448db864b66e30022e139c9fd746738a...,972c8bcff7afd1a24f066f8df12d4c6ea331c1303bb907...


## Paired uncertainty check
Run `python scripts/analyze_results.py` to produce 5,000 paired question bootstrap resamples. The interval reflects sensitivity to these questions. Small, related, authored questions do not support broad statistical or causal claims.

In [4]:
import json
path = ROOT / 'reports' / 'paired_bootstrap.json'
analysis = json.loads(path.read_text()) if path.exists() else {'results': []}
pd.DataFrame(analysis['results'])

,chunk_words,questions,mrr_difference_dense_minus_tfidf,bootstrap_95_low,bootstrap_95_high
0,80,18,0.092593,0.0,0.240741
1,160,18,0.092593,0.0,0.240741
2,220,18,0.092593,0.0,0.240741


## Next independent experiment
Write a hypothesis before tuning. Add new documents and independently annotated questions, freeze a held-out set, and report results separately. Record cold loading, warm query timing, model/dependency versions, and hardware. Explain one failure and one design change backed by evidence.